# 00_config
Central configuration + one-time table setup for the zip pipeline (**copy → unzip → ingest JSON**).

Everything you normally need to change is in the *Settings* cell. Safe to `%run` many times: tables are only created if missing and the control table is only seeded if it is empty.

In [0]:
import re
import uuid
import zipfile
import traceback
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, LongType, TimestampType
)

### Settings

In [0]:
catalog = "iics"
schema  = "casemix"

control_table  = f"{catalog}.{schema}.zip_files_and_json_control_test"   
log_table      = f"{catalog}.{schema}.zip_files_execution_log_test"      # COPY + UNZIP log
load_log_table = f"{catalog}.{schema}.json_large_files_load_log"         # JSON ingestion log

SRC_SYSTEM   = "NPHC"                                                    # value of the src_system column
# SRC_SYSTEM = "HSAR"
EXECUTED_BY  = spark.sql("SELECT current_user()").first()[0]


# e.g. case_data_001_20220605_20220620_20260101_235954.zip  ->  mid_date = 20220620
FILE_NAME_PATTERN = re.compile(
    r"^(?P<table_name>.+?)_(?P<start_date>\d{8})_(?P<mid_date>\d{8})_(?P<end_date>\d{8})_\d{6}\.(zip|json)$",
    re.IGNORECASE,
)

In [0]:
spark.sql(f"drop table if exists {control_table}")
spark.sql(f"drop table if exists {log_table}")
spark.sql(f"drop table if exists {load_log_table}")

### Control table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {control_table} (
    s_no                        INT NOT NULL,
    src_system                  STRING NOT NULL,
    zip_source                  STRING NOT NULL,
    zip_source_path             STRING NOT NULL,
    zip_tgt                     STRING NOT NULL,
    zip_tgt_path                STRING NOT NULL,
    unzip_tgt_path              STRING NOT NULL,
    file_type                   STRING NOT NULL,
    tgt_catalog                 STRING NOT NULL,
    tgt_schema                  STRING NOT NULL,
    tgt_table                   STRING NOT NULL,
    versioning_partition_filed  STRING NOT NULL,
    is_full_load                INT NOT NULL,
    is_incremental_load         INT NOT NULL,
    is_active                   INT NOT NULL
)
USING DELTA
""")

### Log tables
`log_table` records the **COPY** and **UNZIP** stages. `load_log_table` records the **JSON ingestion** (one row per JSON file, including the `zip_file` it came from).

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id",      StringType(),    True),
    StructField("stage",       StringType(),    True),   # COPY | UNZIP
    StructField("s_no",        IntegerType(),   True),
    StructField("source",      StringType(),    True),
    StructField("file_name",   StringType(),    True),
    StructField("source_path", StringType(),    True),
    StructField("target_path", StringType(),    True),
    StructField("status",      StringType(),    True),   # SUCCESS | FAILED | SKIPPED
    StructField("message",     StringType(),    True),
    StructField("start_time",  TimestampType(), True),
    StructField("end_time",    TimestampType(), True),
])

LOAD_LOG_SCHEMA = StructType([
    StructField("run_id",           StringType(),    True),
    StructField("src",              StringType(),    True),
    StructField("src_path",         StringType(),    True),
    StructField("zip_file",         StringType(),    True),
    StructField("file_name",        StringType(),    True),
    StructField("file_path",        StringType(),    True),
    StructField("file_size_mb",     DoubleType(),    True),
    StructField("file_modified_ms", LongType(),      True),
    StructField("target_table",     StringType(),    True),
    StructField("status",           StringType(),    True),   # SUCCESS | FAILED
    StructField("error_type",       StringType(),    True),
    StructField("error_message",    StringType(),    True),
    StructField("started_at",       TimestampType(), True),
    StructField("ended_at",         TimestampType(), True),
    StructField("executed_by",      StringType(),    True),
])

def create_log_table(name, struct):
    cols = ",\n    ".join(f"{f.name} {f.dataType.simpleString().upper()}" for f in struct.fields)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {name} (\n    {cols}\n) USING DELTA")

create_log_table(log_table, LOG_SCHEMA)
create_log_table(load_log_table, LOAD_LOG_SCHEMA)

### control table data

In [0]:
if spark.table(control_table).count() == 0:
    spark.sql(f"""
        INSERT INTO {control_table}
        (s_no,src_system, zip_source, zip_source_path, zip_tgt, zip_tgt_path, unzip_tgt_path, file_type,
         tgt_catalog, tgt_schema, tgt_table, versioning_partition_filed,
         is_full_load, is_incremental_load, is_active)
        VALUES
        (1, 'NPHC','s3', 's3://cittabase-s3-test/Databricks-poc/zip_files/',
         'databricks_volume', '/Volumes/iics/casemix/history/casemix/zip_files/raw/',
         '/Volumes/iics/casemix/history/casemix/zip_files/processed/', 'json',
         'iics', 'casemix', 'json_test_1', 'file_name', 1, 0, 1)
    """)
    print("Seeded control table with 1 row.")
else:
    print("Control table already has rows - not seeded.")

In [0]:
%sql
drop table if exists iics.casemix.json_test_1

In [0]:
%sql
SELECT * FROM iics.casemix.json_test_1
order by file_id